# Day 1-02｜Homography：一點如何被轉到 BEV

> Python「黑客松」實戰分析籃球運動數據  
> Notebook 以初學 Python 學員為主：先觀察、再改變少量變數、最後才串接。

目標：理解 Homography 需要兩邊對應點。先用固定圖片與固定 BEV 圖，體驗「相機座標 → 鳥瞰座標」。

In [ ]:
from pathlib import Path
import sys

# 在 Colab 中先掛載 Drive；本機執行時會自動略過。
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機或 zip 解壓後執行時，從目前資料夾往上找。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

print("COURSE_ROOT =", COURSE_ROOT)
print("assets =", COURSE_ROOT / "assets")

In [ ]:
import numpy as np
from src.cv_utils import (
    load_json,
    read_image_rgb,
    draw_points,
    show_image,
    side_by_side,
    save_image_rgb,
)
from src.geometry_utils import compute_homography, project_points

points_json = COURSE_ROOT / "assets" / "samples" / "sample_homography_points.json"
data = load_json(points_json)

camera_points = data["camera_points"]
bev_points = data["bev_points"]

H = compute_homography(camera_points, bev_points)
print("Homography matrix H =")
print(np.round(H, 3))

In [ ]:
frame = read_image_rgb(COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png")
bev = read_image_rgb(COURSE_ROOT / "assets" / "samples" / "sample_bev_court.png")

frame_vis = draw_points(
    frame, camera_points, labels=[str(i) for i in range(len(camera_points))]
)
bev_vis = draw_points(bev, bev_points, labels=[str(i) for i in range(len(bev_points))])
show_image(side_by_side(frame_vis, bev_vis), "左：相機點；右：BEV 對應點")

接著指定相機畫面中的任一點，例如球員腳底點，看看它在 BEV 上的位置。

In [ ]:
# 你可以修改這個點。
camera_point = [[586, 715]]
bev_point = project_points(camera_point, H)

print("camera point:", camera_point[0])
print("BEV point:", np.round(bev_point[0], 2).tolist())

frame_vis = draw_points(frame, camera_point, labels=["input"])
bev_vis = draw_points(bev, bev_point, labels=["projected"])
out = side_by_side(frame_vis, bev_vis)
show_image(out, "單點投影結果")

save_path = COURSE_ROOT / "assets" / "results" / "d1_02_point_projection.png"
save_image_rgb(save_path, out)
print("saved:", save_path)

小練習：把 `camera_point` 改成另一個球員的腳底位置，觀察 BEV 點會怎麼移動。